<a href="https://colab.research.google.com/github/Odewenu/network-anomaly-detection/blob/main/notebooks/01_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

train = pd.read_parquet("UNSW_NB15_training-set.parquet")
test = pd.read_parquet("UNSW_NB15_testing-set.parquet")

print(train.shape, test.shape)
print(train.columns.tolist())
print(train["label"].value_counts())

OSError: Could not open Parquet input source '<Buffer>': Couldn't deserialize thrift: don't know what type: 


In [2]:
import os

for f in ["UNSW_NB15_training-set.parquet", "UNSW_NB15_testing-set.parquet"]:
    print(f, os.path.getsize(f), "bytes")
    with open(f, "rb") as fh:
        print("starts with:", fh.read(4))

UNSW_NB15_training-set.parquet 15328347 bytes
starts with: b'PAR1'
UNSW_NB15_testing-set.parquet 4538929 bytes
starts with: b'PAR1'


In [3]:
import pandas as pd

for f in ["UNSW_NB15_training-set.parquet", "UNSW_NB15_testing-set.parquet"]:
    try:
        df = pd.read_parquet(f)
        print(f, "OK", df.shape)
    except Exception as e:
        print(f, "FAILED")
        with open(f, "rb") as fh:
            fh.seek(-4, 2)
            print("ends with:", fh.read(4))

UNSW_NB15_training-set.parquet FAILED
ends with: b'PAR1'
UNSW_NB15_testing-set.parquet OK (82332, 36)


In [4]:
import kagglehub, os

path = kagglehub.dataset_download("dhoogla/unswnb15")
print("Downloaded to:", path)

for root, dirs, files in os.walk(path):
    for name in files:
        full = os.path.join(root, name)
        print(full, os.path.getsize(full), "bytes")

Using Colab cache for faster access to the 'unswnb15' dataset.
Downloaded to: /kaggle/input/unswnb15
/kaggle/input/unswnb15/UNSW_NB15_testing-set.parquet 4538929 bytes
/kaggle/input/unswnb15/UNSW_NB15_training-set.parquet 9618034 bytes


In [5]:
import pandas as pd

folder = "/kaggle/input/unswnb15/"
train = pd.read_parquet(folder + "UNSW_NB15_training-set.parquet")
test = pd.read_parquet(folder + "UNSW_NB15_testing-set.parquet")

print(train.shape, test.shape)
print(train.columns.tolist())
print(train["label"].value_counts())

(175341, 36) (82332, 36)
['dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'is_sm_ips_ports', 'attack_cat', 'label']
label
1    119341
0     56000
Name: count, dtype: int64


In [6]:
print("Missing values in train:", train.isnull().sum().sum())
print("Missing values in test:", test.isnull().sum().sum())
print("Duplicate rows in train:", train.duplicated().sum())
print("Duplicate rows in test:", test.duplicated().sum())
print(train["attack_cat"].value_counts())

Missing values in train: 0
Missing values in test: 0
Duplicate rows in train: 78519
Duplicate rows in test: 32361
attack_cat
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Analysis           2000
Backdoor           1746
Shellcode          1133
Worms               130
Name: count, dtype: int64


In [7]:
train = train.drop_duplicates().reset_index(drop=True)
test = test.drop_duplicates().reset_index(drop=True)

print("Train shape after:", train.shape)
print("Test shape after:", test.shape)
print(train["label"].value_counts())
print(train["attack_cat"].value_counts())

Train shape after: (96822, 36)
Test shape after: (49971, 36)
label
0    48894
1    47928
Name: count, dtype: int64
attack_cat
Normal            48894
Exploits          19360
Fuzzers           14082
Reconnaissance     6000
DoS                3369
Generic            1800
Backdoor           1121
Analysis           1119
Shellcode           954
Worms               123
Name: count, dtype: int64


In [8]:
for col in ["proto", "service", "state"]:
    print(col, "->", train[col].nunique(), "unique values")
    print(train[col].value_counts().head(5))
    print()

num_cols = train.select_dtypes(include="number").columns.drop("label")
print(train[num_cols].skew().sort_values(ascending=False).head(10))

proto -> 133 unique values
proto
tcp     75805
udp     15238
ospf      769
arp       591
unas      350
Name: count, dtype: int64

service -> 13 unique values
service
-           57971
http        18090
dns          7181
smtp         5023
ftp-data     3282
Name: count, dtype: int64

state -> 9 unique values
state
FIN    74345
CON    10976
INT    10812
REQ      594
RST       83
Name: count, dtype: int64

trans_depth          129.987103
response_body_len     57.158455
sbytes                33.744322
sloss                 33.374062
dloss                 31.992615
dbytes                30.509004
spkts                 30.431532
dinpkt                29.005125
dpkts                 28.606046
djit                  25.801117
dtype: float64


In [9]:
import numpy as np

# 1. Log transform the heavily skewed number columns
num_cols = train.select_dtypes(include="number").columns.drop("label")
skew = train[num_cols].skew()
skewed_cols = [c for c in skew[skew > 5].index
               if train[c].min() >= 0 and test[c].min() >= 0
               and train[c].nunique() > 2]

for df in (train, test):
    df[skewed_cols] = np.log1p(df[skewed_cols])

print("Log-transformed", len(skewed_cols), "columns")

# 2. Group rare protocols into "other"
top_proto = train["proto"].value_counts().head(5).index
for df in (train, test):
    df["proto"] = df["proto"].where(df["proto"].isin(top_proto), "other")

# 3. Turn text columns into numbers
cat_cols = ["proto", "service", "state"]
train = pd.get_dummies(train, columns=cat_cols, dtype=int)
test = pd.get_dummies(test, columns=cat_cols, dtype=int)
test = test.reindex(columns=train.columns, fill_value=0)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print(train[skewed_cols].skew().sort_values(ascending=False).head(5))

Log-transformed 21 columns


TypeError: Cannot setitem on a Categorical with a new category (other), set the categories first

In [10]:
# Convert text columns to plain text
cat_cols = ["proto", "service", "state"]
for df in (train, test):
    for c in cat_cols:
        df[c] = df[c].astype(str)

# Group rare protocols into "other"
top_proto = train["proto"].value_counts().head(5).index
for df in (train, test):
    df["proto"] = df["proto"].where(df["proto"].isin(top_proto), "other")

# Turn text columns into numbers
train = pd.get_dummies(train, columns=cat_cols, dtype=int)
test = pd.get_dummies(test, columns=cat_cols, dtype=int)
test = test.reindex(columns=train.columns, fill_value=0)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print(train[skewed_cols].skew().sort_values(ascending=False).head(5))

Train shape: (96822, 61)
Test shape: (49971, 61)
response_body_len    2.965966
synack               2.719262
dur                  2.703959
trans_depth          1.926722
dloss                1.195676
dtype: float64


/usr/local/lib/python3.13/dist-packages/pandas/core/nanops.py:1487: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)


In [11]:
from sklearn.preprocessing import StandardScaler
import joblib, os

train["attack_cat"] = train["attack_cat"].astype(str)
test["attack_cat"] = test["attack_cat"].astype(str)

# Only scale real number columns (not label, attack_cat, or 0/1 columns)
to_scale = [c for c in train.columns
            if c not in ("label", "attack_cat") and train[c].nunique() > 2]

scaler = StandardScaler()
train[to_scale] = scaler.fit_transform(train[to_scale])
test[to_scale] = scaler.transform(test[to_scale])

print("Scaled", len(to_scale), "columns")
print(train[to_scale].mean().round(2).head())

# Save everything
os.makedirs("processed", exist_ok=True)
train.to_csv("processed/train_clean.csv", index=False)
test.to_csv("processed/test_clean.csv", index=False)
joblib.dump(scaler, "processed/scaler.pkl")

!zip -r processed.zip processed
from google.colab import files
files.download("processed.zip")

Scaled 30 columns
dur       0.0
spkts     0.0
dpkts    -0.0
sbytes    0.0
dbytes    0.0
dtype: float64
  adding: processed/ (stored 0%)
  adding: processed/test_clean.csv (deflated 79%)
  adding: processed/train_clean.csv (deflated 79%)
  adding: processed/scaler.pkl (deflated 26%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
import os, shutil

os.makedirs("data/processed", exist_ok=True)

# make the files smaller so GitHub accepts them
def shrink(df):
    df = df.copy()
    f = df.select_dtypes("float64").columns
    df[f] = df[f].astype("float32")
    return df

shrink(train).to_parquet("data/processed/train_clean.parquet", index=False, compression="gzip")
shrink(test).to_parquet("data/processed/test_clean.parquet", index=False, compression="gzip")
shutil.copy("processed/scaler.pkl", "data/processed/scaler.pkl")

# a simple list of all columns (the "data schema" the PDF asks for)
schema = pd.DataFrame({"column": train.columns, "type": train.dtypes.astype(str).values})
schema.to_csv("data/processed/data_schema.csv", index=False)

for f in os.listdir("data/processed"):
    print(f, round(os.path.getsize("data/processed/" + f) / 1e6, 1), "MB")

!zip -r data.zip data
from google.colab import files
files.download("data.zip")

train_clean.parquet 6.3 MB
data_schema.csv 0.0 MB
scaler.pkl 0.0 MB
test_clean.parquet 3.3 MB
  adding: data/ (stored 0%)
  adding: data/processed/ (stored 0%)
  adding: data/processed/train_clean.parquet (deflated 1%)
  adding: data/processed/data_schema.csv (deflated 69%)
  adding: data/processed/scaler.pkl (deflated 26%)
  adding: data/processed/test_clean.parquet (deflated 1%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>